# Toxic Comment Detection — Recommend Moderation Action

**Pipeline stage**: `recommend-moderation-action`  
**Upstream**: `assess-severity` → produces `assess_severity_output.json`  
**Downstream**: Gradio UI / business user display → receives `moderation_action_output.json`

This notebook turns the severity assessment into a concrete moderation recommendation that a non-technical platform operator can act on.

## 1. Setup Paths and Imports

In [ ]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Any

from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()

def detect_project_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    markers = ["assess_severity_output.json", "run_inference_output.json", "raw_data"]
    for candidate in candidates:
        if sum((candidate / marker).exists() for marker in markers) >= 2:
            return candidate
    raise FileNotFoundError("Could not automatically detect project root from the current notebook location.")

PROJECT_ROOT = detect_project_root(NOTEBOOK_DIR)
ASSESS_SEVERITY_OUTPUT_PATH = PROJECT_ROOT / "assess_severity_output.json"
OUTPUT_PATH = PROJECT_ROOT / "moderation_action_output.json"

print(f"Project root                   : {PROJECT_ROOT}")
print(f"Assess severity state path     : {ASSESS_SEVERITY_OUTPUT_PATH}")
print(f"Moderation action output path  : {OUTPUT_PATH}")

## 2. Helper Functions

In [ ]:
def load_json(path: Path) -> Dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"Required JSON not found: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def recommend_action(severity_state: Dict[str, Any]) -> Dict[str, Any]:
    severity_label = severity_state["severity_label"]
    review_priority = severity_state["review_priority"]
    escalation_required = bool(severity_state["escalation_required"])
    toxicity_probability = float(severity_state["toxicity_probability"])
    confidence = float(severity_state["confidence"])
    comment_text = severity_state["comment_text"]

    if severity_label == "clean":
        action_code = "allow"
        action_label = "Allow Comment"
        action_priority = "none"
        human_review_required = False
        user_notification = "none"
        moderator_rationale = "The comment is comfortably below the toxic threshold and does not need intervention."
        business_message = "No action is recommended. The comment can remain visible."
    elif severity_label == "borderline":
        action_code = "allow_with_monitoring"
        action_label = "Allow but Monitor"
        action_priority = "low"
        human_review_required = False
        user_notification = "none"
        moderator_rationale = "The comment is currently non-toxic but sits close to the decision boundary, so lightweight monitoring is appropriate."
        business_message = "Allow the comment to remain visible, but keep it in a low-priority monitoring queue."
    elif severity_label == "low":
        action_code = "soft_warn"
        action_label = "Soft Warning"
        action_priority = "low"
        human_review_required = True
        user_notification = "gentle_warning"
        moderator_rationale = "The content crossed the toxicity threshold weakly, so a softer response is safer than immediate removal."
        business_message = "Queue the comment for review and consider sending the user a soft civility warning."
    elif severity_label == "medium":
        action_code = "review_and_warn"
        action_label = "Review and Warn"
        action_priority = "medium"
        human_review_required = True
        user_notification = "warning"
        moderator_rationale = "The model indicates meaningful toxicity, but the evidence is not strong enough to justify automatic removal without review."
        business_message = "Send the case to a moderator review queue and issue a warning if the comment violates policy."
    elif severity_label == "high":
        action_code = "hide_and_review"
        action_label = "Hide and Review"
        action_priority = "high"
        human_review_required = True
        user_notification = "warning"
        moderator_rationale = "The comment has a strong toxic signal and should be hidden quickly while waiting for moderator confirmation."
        business_message = "Temporarily hide the comment and send it for high-priority moderator review."
    else:
        action_code = "remove_and_escalate"
        action_label = "Remove and Escalate"
        action_priority = "urgent"
        human_review_required = True
        user_notification = "final_warning"
        moderator_rationale = "The comment shows a very strong toxic signal and should be removed immediately with escalation to urgent moderator attention."
        business_message = "Remove the comment immediately and escalate the case to the urgent moderation queue."

    final_recommendation = (
        f"Recommended action: {action_label}. "
        f"Severity is {severity_label} with toxicity probability {toxicity_probability:.4f} "
        f"and confidence {confidence:.4f}."
    )

    ui_explanation = (
        f"This comment was assessed as {severity_label}. {business_message}"
    )

    return {
        "comment_text": comment_text,
        "selected_model_id": severity_state["selected_model_id"],
        "selected_model_label": severity_state.get("selected_model_label"),
        "predicted_label": severity_state["predicted_label"],
        "predicted_class_id": severity_state["predicted_class_id"],
        "is_toxic": severity_state["is_toxic"],
        "toxicity_probability": round(toxicity_probability, 4),
        "non_toxic_probability": round(float(severity_state["non_toxic_probability"]), 4),
        "confidence": round(confidence, 4),
        "threshold_used": float(severity_state["threshold_used"]),
        "severity_label": severity_label,
        "severity_rank": severity_state["severity_rank"],
        "review_priority": review_priority,
        "escalation_required": escalation_required,
        "action_code": action_code,
        "action_label": action_label,
        "action_priority": action_priority,
        "human_review_required": human_review_required,
        "user_notification": user_notification,
        "moderator_rationale": moderator_rationale,
        "business_message": business_message,
        "final_recommendation": final_recommendation,
        "ui_explanation": ui_explanation,
        "summary_for_moderator": severity_state["summary_for_moderator"],
        "severity_explanation": severity_state["severity_explanation"],
        "selection_justification": severity_state.get("selection_justification"),
        "bias_assessment": severity_state.get("bias_assessment"),
        "inference_timestamp_utc": severity_state.get("inference_timestamp_utc"),
        "severity_assessed_at_utc": severity_state.get("severity_assessed_at_utc"),
        "action_recommended_at_utc": datetime.now(timezone.utc).isoformat()
    }


## 3. Load Upstream State

In [ ]:
severity_state = load_json(ASSESS_SEVERITY_OUTPUT_PATH)
display(severity_state)

## 4. Recommend Moderation Action

In [ ]:
moderation_action_output = recommend_action(severity_state)
display(moderation_action_output)

## 5. Write Output to Agent State

In [ ]:
with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(moderation_action_output, f, indent=4, ensure_ascii=False)

print(f"Saved moderation action output to: {OUTPUT_PATH}")

## 6. Business-Facing Output Contract

The Gradio UI or business-facing layer should primarily display these fields from `moderation_action_output.json`:

- `action_label`: final action recommendation for the platform team
- `business_message`: short business-facing action summary
- `ui_explanation`: plain-language explanation suitable for a non-technical user
- `severity_label`: severity band that explains the recommendation
- `review_priority`: moderator queue priority
- `human_review_required`: whether the case must be sent to a human moderator
- `user_notification`: whether the user should receive no message, a gentle warning, a warning, or a final warning
- `final_recommendation`: one-line recommendation string for the UI
